In [1]:
import os
import re
import random
import numpy as np
import pandas as pd

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN

/Users/peichenxu/bertopic-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [3]:
file1 = "1-Questionaire-Tourist.xlsx"
file2 = "2-FDH-Onsite.xlsx"
file3 = "3-30pcp.xlsx"

df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
df3 = pd.read_excel(file3)

# File 1 uses translated English text
df1_clean = pd.DataFrame({
    "source": "tourist_questionnaire",
    "participant_id": df1["id"],
    "question": df1["question"],
    "text": df1["text_ENG"]
})

# File 2
df2_clean = pd.DataFrame({
    "source": "fdh_onsite",
    "participant_id": df2["FDH_id"],
    "question": None,
    "text": df2["text"]
})

# File 3
df3_clean = pd.DataFrame({
    "source": "pcp_30",
    "participant_id": df3["Participant_id"],
    "question": df3["question"],
    "text": df3["text"]
})

combined_df = pd.concat(
    [df1_clean, df2_clean, df3_clean],
    ignore_index=True
)

combined_df = combined_df.dropna(subset=["text"]).copy()
combined_df["text"] = combined_df["text"].astype(str).str.strip()
combined_df = combined_df[combined_df["text"] != ""].copy()

print(combined_df["source"].value_counts())
print("Total documents:", len(combined_df))

combined_df.head()

source
pcp_30                   30
tourist_questionnaire    27
fdh_onsite               23
Name: count, dtype: int64
Total documents: 80


,source,participant_id,question,text
0,tourist_questionnaire,1,对香港声环境和外佣聚集的看法,I once stayed in a hotel in Mong Kok for a nig...
1,tourist_questionnaire,2,对香港声环境和外佣聚集的看法,"Separating activity and quiet, with each perso..."
2,tourist_questionnaire,3,对香港声环境和外佣聚集的看法,Foreign domestic helpers only gather in specif...
3,tourist_questionnaire,4,对香港声环境和外佣聚集的看法,"The first time, I felt the domestic helpers en..."
4,tourist_questionnaire,5,对香港声环境和外佣聚集的看法,"Due to cultrual differences, people from the M..."


In [57]:
def normalize_terms(text):
    text = str(text)

    replacements = {
        # Foreign domestic helper terms
        r"\bforeign domestic helpers\b": "foreign_domestic_helper",
        r"\bforeign domestic helper\b": "foreign_domestic_helper",
        r"\bdomestic helpers\b": "foreign_domestic_helper",
        r"\bdomestic helper\b": "foreign_domestic_helper",
        r"\bdomestic workers\b": "foreign_domestic_helper",
        r"\bdomestic worker\b": "foreign_domestic_helper",
        r"\bfdhs\b": "foreign_domestic_helper",
        r"\bfdh\b": "foreign_domestic_helper",
        r"\bhelpers\b": "foreign_domestic_helper",
        r"\bhelper\b": "foreign_domestic_helper",
        r"\bmaids\b": "foreign_domestic_helper",
        r"\bmaid\b": "foreign_domestic_helper",
        r"\bfilipino helpers\b": "foreign_domestic_helper",
        r"\bfilipino helper\b": "foreign_domestic_helper",
        r"\bfilipino maids\b": "foreign_domestic_helper",
        r"\bfilipino maid\b": "foreign_domestic_helper",
        r"\bmigrant workers\b": "foreign_domestic_helper",
        
        # Soundscape term normalization
        r"\bsounds\b": "sound",
        r"\bnoises\b": "noise",
        r"\benvironments\b": "environment",
        r"\bplaces\b": "place",
        r"\bparks\b": "park",
        r"\bweekends\b": "weekend",
        r"\bweekdays\b": "weekday",

        # Location normalization
        r"\bklt\b": "kowloon_tong",
        r"\bkowloon tong\b": "kowloon_tong",
        r"\bmk\b": "mong_kok",
        r"\bmong kok\b": "mong_kok",
        r"\bhong kong\b": "hong_kong",
        r"\bh\.?\s*k\.?\b": "hong_kong",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text

In [58]:
custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    # Basic grammar words, added explicitly for clarity
    "the", "a", "an", "is", "are", "was", "were",
    "in", "on", "at", "to", "from", "of",
    "and", "but", "or", "which", "for", "with",
    "as", "by", "this", "that", "these", "those",
    "it", "its", "they", "them", "their", "there",
    
    # Filler words
    "yeah", "yes", "like", "think", "maybe", "actually", "really",
    "quite", "just", "lot", "lots", "kind", "thing", "things",
    "something", "sometimes", "probably", "basically",
    
    # Transcript / speaking filler
    "feel", "felt", "feels", "say", "said", "says",
    "see", "seen", "look", "looks", "looked",
    "watch", "watched", "video", "clip", "clips", "review",
    
    # Common contractions / token fragments
    "doesn", "don", "isn", "aren", "wasn", "weren",
    "didn", "couldn", "wouldn", "shouldn",
    "ve", "ll", "re", "im", "i'm"
})

In [59]:
def clean_text(text):
    text = normalize_terms(text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove bracketed interviewer prompts
    text = re.sub(r"\[.*?\]", " ", text)

    # Keep English letters and underscores
    text = re.sub(r"[^a-zA-Z_\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    tokens = [
        token for token in tokens
        if token not in custom_stopwords
        and len(token) > 2
    ]

    return " ".join(tokens)

In [60]:
combined_df["clean_text"] = combined_df["text"].apply(clean_text)

docs = combined_df["clean_text"].tolist()

combined_df[["source", "participant_id", "text", "clean_text"]].head()

,source,participant_id,text,clean_text
0,tourist_questionnaire,1,I once stayed in a hotel in Mong Kok for a nig...,stayed hotel mong_kok night windows open hear ...
1,tourist_questionnaire,2,"Separating activity and quiet, with each perso...",separating activity quiet person taking need
2,tourist_questionnaire,3,Foreign domestic helpers only gather in specif...,foreign_domestic_helper gather specific place ...
3,tourist_questionnaire,4,"The first time, I felt the domestic helpers en...",time foreign_domestic_helper enriched environm...
4,tourist_questionnaire,5,"Due to cultrual differences, people from the M...",cultrual differences people mainland understan...


In [61]:
from collections import Counter

all_words = " ".join(docs).split()
word_counts = Counter(all_words)

word_counts.most_common(50)

[('sound', 118),
 ('park', 80),
 ('people', 72),
 ('footbridge', 56),
 ('noise', 51),
 ('weekday', 44),
 ('foreign_domestic_helper', 43),
 ('mong_kok', 39),
 ('soundscape', 34),
 ('noisy', 32),
 ('area', 32),
 ('kowloon_tong', 31),
 ('weekend', 27),
 ('traffic', 25),
 ('environment', 24),
 ('music', 23),
 ('quiet', 22),
 ('crowded', 22),
 ('hong_kong', 22),
 ('sunday', 22),
 ('living', 19),
 ('sundays', 19),
 ('hear', 18),
 ('human', 18),
 ('place', 17),
 ('different', 17),
 ('district', 17),
 ('sun', 17),
 ('birds', 15),
 ('talking', 15),
 ('sat', 15),
 ('construction', 14),
 ('chatting', 14),
 ('gathering', 14),
 ('city', 13),
 ('bridge', 13),
 ('walking', 13),
 ('time', 12),
 ('compared', 12),
 ('saturdays', 12),
 ('activities', 11),
 ('loud', 11),
 ('natural', 11),
 ('usually', 10),
 ('friends', 10),
 ('day', 10),
 ('footsteps', 10),
 ('saturday', 10),
 ('cars', 9),
 ('live', 9)]

In [62]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    docs,
    show_progress_bar=True,
    normalize_embeddings=True
)

np.save("combined_embeddings.npy", embeddings)

Batches: 100%|██████████| 3/3 [00:00<00:00,  5.77it/s]


In [47]:
embeddings = np.load("combined_embeddings.npy")

In [68]:
umap_model = UMAP(
    n_neighbors=6,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=4,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    token_pattern=r"(?u)\b[a-zA-Z_][a-zA-Z_]+\b"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

combined_df["topic"] = topics

topic_info = topic_model.get_topic_info()
topic_info

2026-05-15 15:01:43,821 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-15 15:01:44,013 - BERTopic - Dimensionality - Completed ✓
2026-05-15 15:01:44,015 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-15 15:01:44,052 - BERTopic - Cluster - Completed ✓
2026-05-15 15:01:44,068 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-15 15:01:44,193 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,9,-1_walking_people chatting_mom_kowloon_tong,"[walking, people chatting, mom, kowloon_tong, ...",[park silent footbridge noisy crowded saturday...
1,0,24,0_footbridge_mong_kok_area_soundscape,"[footbridge, mong_kok, area, soundscape, kowlo...",[soundscape park footbridge lively vibing area...
2,1,12,1_music_birds_kowloon_tong_friends,"[music, birds, kowloon_tong, friends, good, ni...",[chose kln mong_kok park comparatively quiet k...
3,2,11,2_hong_kong_city_environment_time,"[hong_kong, city, environment, time, gather, u...",[gathering activities foreign_domestic_helper ...
4,3,11,3_sound environment_hong_kong_central_environment,"[sound environment, hong_kong, central, enviro...",[hong_kong sound environment significant impac...
5,4,8,4_quiet_chatter_loud_footbridge noisy,"[quiet, chatter, loud, footbridge noisy, footb...",[park quieter peaceful footbridge sound birds ...
6,5,5,5_far_annoying_right_rock,"[far, annoying, right, rock, usually, hear, dr...",[annoying moderate noisy coming outside inside...


In [65]:
# Save document-level topic assignments
combined_df.to_excel(
    "combined_documents_with_final_topics.xlsx",
    index=False
)

# Save topic-level summary
topic_info.to_csv(
    "combined_final_topic_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved files:")
print("combined_documents_with_final_topics.xlsx")
print("combined_final_topic_summary.csv")

Saved files:
combined_documents_with_final_topics.xlsx
combined_final_topic_summary.csv


In [70]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

analyzer = SentimentIntensityAnalyzer()

# Add BERTopic topic labels to your combined dataframe
combined_df["topic"] = topics

# Optional: remove BERTopic outlier topic -1
df_sentiment = combined_df[combined_df["topic"] != -1].copy()

# Apply VADER to the original text column
vader_scores = df_sentiment["text"].apply(
    lambda x: analyzer.polarity_scores(str(x))
)

vader_df = pd.DataFrame(list(vader_scores))

df_sentiment = pd.concat(
    [df_sentiment.reset_index(drop=True), vader_df.reset_index(drop=True)],
    axis=1
)

df_sentiment.head()

,source,participant_id,question,text,clean_text,topic,neg,neu,pos,compound
0,tourist_questionnaire,1,对香港声环境和外佣聚集的看法,I once stayed in a hotel in Mong Kok for a nig...,stayed hotel mong_kok night windows open hear ...,1,0.035,0.917,0.048,0.1265
1,tourist_questionnaire,2,对香港声环境和外佣聚集的看法,"Separating activity and quiet, with each perso...",separating activity quiet person taking need,4,0.000,1.000,0.000,0.0000
2,tourist_questionnaire,3,对香港声环境和外佣聚集的看法,Foreign domestic helpers only gather in specif...,foreign_domestic_helper gather specific place ...,2,0.000,0.925,0.075,0.2732
3,tourist_questionnaire,4,对香港声环境和外佣聚集的看法,"The first time, I felt the domestic helpers en...",time foreign_domestic_helper enriched environm...,2,0.092,0.771,0.137,0.4378
4,tourist_questionnaire,5,对香港声环境和外佣聚集的看法,"Due to cultrual differences, people from the M...",cultrual differences people mainland understan...,2,0.075,0.727,0.198,0.5535


In [71]:
def sentiment_label(compound):
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_sentiment["sentiment"] = df_sentiment["compound"].apply(sentiment_label)

In [72]:
topic_sentiment = (
    df_sentiment
    .groupby("topic")
    .agg(
        doc_count=("text", "count"),
        mean_compound=("compound", "mean"),
        mean_positive=("pos", "mean"),
        mean_neutral=("neu", "mean"),
        mean_negative=("neg", "mean")
    )
    .reset_index()
)

topic_sentiment["overall_sentiment"] = topic_sentiment["mean_compound"].apply(sentiment_label)

topic_sentiment

,topic,doc_count,mean_compound,mean_positive,mean_neutral,mean_negative,overall_sentiment
0,0,24,0.707621,0.134375,0.842708,0.022875,Positive
1,1,12,0.622808,0.161167,0.802000,0.036833,Positive
2,2,11,0.544782,0.165909,0.802182,0.031818,Positive
3,3,11,0.039355,0.087909,0.841455,0.070727,Neutral
4,4,8,0.098025,0.026250,0.922250,0.051625,Positive
5,5,5,-0.234880,0.065200,0.809400,0.125400,Negative


In [74]:
sentiment_counts = (
    df_sentiment
    .groupby(["topic", "sentiment"])
    .size()
    .reset_index(name="count")
    .pivot(index="topic", columns="sentiment", values="count")
    .fillna(0)
)

sentiment_counts

sentiment,Negative,Neutral,Positive
topic,,,
0,1.0,0.0,23.0
1,0.0,0.0,12.0
2,0.0,0.0,11.0
3,5.0,2.0,4.0
4,2.0,3.0,3.0
5,4.0,0.0,1.0


In [75]:
for topic_id in sorted(df_sentiment["topic"].unique()):
    subset = df_sentiment[df_sentiment["topic"] == topic_id]
    
    print("=" * 80)
    print("Topic:", topic_id)
    print("Mean compound:", subset["compound"].mean())
    
    print("\nMost negative comments:")
    display(
        subset.sort_values("compound").head(5)[
            ["source", "participant_id", "text", "compound", "sentiment"]
        ]
    )
    
    print("\nMost positive comments:")
    display(
        subset.sort_values("compound", ascending=False).head(5)[
            ["source", "participant_id", "text", "compound", "sentiment"]
        ]
    )

Topic: 0
Mean compound: 0.7076208333333334

Most negative comments:


,source,participant_id,text,compound,sentiment
57,pcp_30,14,"On the footbridge (MK), sound of cars passing ...",-0.3815,Negative
31,fdh_onsite,KLT-2-FDH-Joyce,[In Central or in Mong Kok] There is also musi...,0.1974,Positive
64,pcp_30,21,the park has more natural sounds such as birds...,0.2633,Positive
15,tourist_questionnaire,19,I hope tourists can follow the rules.,0.4404,Positive
17,tourist_questionnaire,21,"There are no unified venues for them, such as ...",0.4434,Positive



Most positive comments:


,source,participant_id,text,compound,sentiment
60,pcp_30,17,I chose...\nMK-1-Sat 20250315 125639 00 004\n...,0.9773,Positive
66,pcp_30,26,MK footbridge: Mainly are normal ambience soun...,0.9749,Positive
50,pcp_30,7,"Based on the KLT-1 clips, the park and footbri...",0.9738,Positive
56,pcp_30,13,The soundscape of the park and footbridge is r...,0.9731,Positive
48,pcp_30,5,I chose KLT1 and MK1\nThe soundscape of the pa...,0.9659,Positive


Topic: 1
Mean compound: 0.6228083333333333

Most negative comments:


...

,source,participant_id,text,compound,sentiment
49,pcp_30,6,"The footbridge noisy compared to the park, and...",0.6744,Positive
69,pcp_30,29,The park is quieter and more peaceful than the...,0.6461,Positive
62,pcp_30,19,The soundscape of the footbridge is very noisy...,0.4213,Positive
46,pcp_30,3,The soundscape of the park and the footbridge ...,0.0276,Neutral
1,tourist_questionnaire,2,"Separating activity and quiet, with each perso...",0.0000,Neutral


Topic: 5
Mean compound: -0.23487999999999998

Most negative comments:


,source,participant_id,text,compound,sentiment
33,fdh_onsite,KLT-1-FDH-Shenna,"You see, sometimes it's annoying, sometimes it...",-0.8052,Negative
51,pcp_30,8,Very noisy\nSome of them are very annoying wit...,-0.6106,Negative
42,fdh_onsite,MK-1-Yui-Bella,[My favorite sound] seems like… there is. Gene...,-0.3306,Negative
20,tourist_questionnaire,25,"It's a bit noisy, but it enriches the city's s...",-0.0900,Negative
30,fdh_onsite,KLT-2-FDH-Joyce,[I feel about Hong Kong and Philippine] that i...,0.6620,Positive



Most positive comments:


,source,participant_id,text,compound,sentiment
30,fdh_onsite,KLT-2-FDH-Joyce,[I feel about Hong Kong and Philippine] that i...,0.6620,Positive
20,tourist_questionnaire,25,"It's a bit noisy, but it enriches the city's s...",-0.0900,Negative
42,fdh_onsite,MK-1-Yui-Bella,[My favorite sound] seems like… there is. Gene...,-0.3306,Negative
51,pcp_30,8,Very noisy\nSome of them are very annoying wit...,-0.6106,Negative
33,fdh_onsite,KLT-1-FDH-Shenna,"You see, sometimes it's annoying, sometimes it...",-0.8052,Negative


In [76]:
sentiment_examples = []

for topic_id in sorted(df_sentiment["topic"].unique()):
    subset = df_sentiment[df_sentiment["topic"] == topic_id].copy()
    
    # Most negative 5 comments
    negative_examples = subset.sort_values("compound").head(5).copy()
    negative_examples["example_type"] = "Most negative"
    
    # Most positive 5 comments
    positive_examples = subset.sort_values("compound", ascending=False).head(5).copy()
    positive_examples["example_type"] = "Most positive"
    
    # Combine
    examples = pd.concat([negative_examples, positive_examples], ignore_index=True)
    
    sentiment_examples.append(examples)

# Combine all topics into one dataframe
sentiment_examples_df = pd.concat(sentiment_examples, ignore_index=True)

# Select useful columns
sentiment_examples_df = sentiment_examples_df[
    [
        "topic",
        "example_type",
        "source",
        "participant_id",
        "text",
        "compound",
        "sentiment"
    ]
]

sentiment_examples_df.head()

,topic,example_type,source,participant_id,text,compound,sentiment
0,0,Most negative,pcp_30,14,"On the footbridge (MK), sound of cars passing ...",-0.3815,Negative
1,0,Most negative,fdh_onsite,KLT-2-FDH-Joyce,[In Central or in Mong Kok] There is also musi...,0.1974,Positive
2,0,Most negative,pcp_30,21,the park has more natural sounds such as birds...,0.2633,Positive
3,0,Most negative,tourist_questionnaire,19,I hope tourists can follow the rules.,0.4404,Positive
4,0,Most negative,tourist_questionnaire,21,"There are no unified venues for them, such as ...",0.4434,Positive


In [77]:
sentiment_examples_df.to_excel(
    "most_positive_negative_comments_by_topic.xlsx",
    index=False
)